In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/march-machine-learning-mania-2026/Conferences.csv
/kaggle/input/competitions/march-machine-learning-mania-2026/WNCAATourneyDetailedResults.csv
/kaggle/input/competitions/march-machine-learning-mania-2026/WRegularSeasonCompactResults.csv
/kaggle/input/competitions/march-machine-learning-mania-2026/MNCAATourneySeedRoundSlots.csv
/kaggle/input/competitions/march-machine-learning-mania-2026/MRegularSeasonDetailedResults.csv
/kaggle/input/competitions/march-machine-learning-mania-2026/MNCAATourneyCompactResults.csv
/kaggle/input/competitions/march-machine-learning-mania-2026/MGameCities.csv
/kaggle/input/competitions/march-machine-learning-mania-2026/WSecondaryTourneyCompactResults.csv
/kaggle/input/competitions/march-machine-learning-mania-2026/WGameCities.csv
/kaggle/input/competitions/march-machine-learning-mania-2026/MSeasons.csv
/kaggle/input/competitions/march-machine-learning-mania-2026/WNCAATourneySlots.csv
/kaggle/input/competitions/march-machine-learning

## EDA

In [2]:
import pandas as pd
import numpy as np

BASE = '/kaggle/input/competitions/march-machine-learning-mania-2026/'

# --- MEN'S FILES ---
m_teams    = pd.read_csv(BASE + 'MTeams.csv')
m_seasons  = pd.read_csv(BASE + 'MSeasons.csv')
m_results  = pd.read_csv(BASE + 'MRegularSeasonCompactResults.csv')
m_detailed = pd.read_csv(BASE + 'MRegularSeasonDetailedResults.csv')
m_tourney  = pd.read_csv(BASE + 'MNCAATourneyCompactResults.csv')
m_seeds    = pd.read_csv(BASE + 'MNCAATourneySeeds.csv')
m_massey   = pd.read_csv(BASE + 'MMasseyOrdinals.csv')

# --- WOMEN'S FILES ---
w_teams    = pd.read_csv(BASE + 'WTeams.csv')
w_seasons  = pd.read_csv(BASE + 'WSeasons.csv')
w_results  = pd.read_csv(BASE + 'WRegularSeasonCompactResults.csv')
w_detailed = pd.read_csv(BASE + 'WRegularSeasonDetailedResults.csv')
w_tourney  = pd.read_csv(BASE + 'WNCAATourneyCompactResults.csv')
w_seeds    = pd.read_csv(BASE + 'WNCAATourneySeeds.csv')

# --- SHARED FILES ---
cities      = pd.read_csv(BASE + 'Cities.csv')
conferences = pd.read_csv(BASE + 'Conferences.csv')

# --- SUBMISSION ---
submission  = pd.read_csv(BASE + 'SampleSubmissionStage2.csv')

print("✅ All files loaded successfully!")
print()
print("=" * 50)
print("MEN'S DATA")
print("=" * 50)
print(f"  Teams:                  {len(m_teams):,}")
print(f"  Regular season games:   {len(m_results):,}")
print(f"  Tournament games:       {len(m_tourney):,}")
print(f"  Seasons (regular):      {m_results['Season'].min()} → {m_results['Season'].max()}")

print()
print("=" * 50)
print("WOMEN'S DATA")
print("=" * 50)
print(f"  Teams:                  {len(w_teams):,}")
print(f"  Regular season games:   {len(w_results):,}")
print(f"  Tournament games:       {len(w_tourney):,}")
print(f"  Seasons (regular):      {w_results['Season'].min()} → {w_results['Season'].max()}")

✅ All files loaded successfully!

MEN'S DATA
  Teams:                  381
  Regular season games:   198,577
  Tournament games:       2,585
  Seasons (regular):      1985 → 2026

WOMEN'S DATA
  Teams:                  379
  Regular season games:   142,507
  Tournament games:       1,717
  Seasons (regular):      1998 → 2026


In [3]:
print("=" * 50)
print("SUBMISSION FILE")
print("=" * 50)
print(submission.head(10))
print()
print(f"Total predictions needed: {len(submission):,}")
print()

# Break apart the ID column to understand it
submission['Season'] = submission['ID'].apply(lambda x: int(x.split('_')[0]))
submission['Team1']  = submission['ID'].apply(lambda x: int(x.split('_')[1]))
submission['Team2']  = submission['ID'].apply(lambda x: int(x.split('_')[2]))

# Men's teams are 1000-1999, Women's are 3000-3999
mens_rows   = submission[submission['Team1'] < 2000]
womens_rows = submission[submission['Team1'] >= 3000]

print(f"Men's matchups to predict:   {len(mens_rows):,}")
print(f"Women's matchups to predict: {len(womens_rows):,}")
print()

# How many unique teams in submission?
all_teams = pd.concat([submission['Team1'], submission['Team2']]).unique()
mens_teams_sub   = [t for t in all_teams if t < 2000]
womens_teams_sub = [t for t in all_teams if t >= 3000]

print(f"Unique men's teams in submission:   {len(mens_teams_sub)}")
print(f"Unique women's teams in submission: {len(womens_teams_sub)}")

SUBMISSION FILE
               ID  Pred
0  2026_1101_1102   0.5
1  2026_1101_1103   0.5
2  2026_1101_1104   0.5
3  2026_1101_1105   0.5
4  2026_1101_1106   0.5
5  2026_1101_1107   0.5
6  2026_1101_1108   0.5
7  2026_1101_1110   0.5
8  2026_1101_1111   0.5
9  2026_1101_1112   0.5

Total predictions needed: 132,133

Men's matchups to predict:   66,430
Women's matchups to predict: 65,703

Unique men's teams in submission:   365
Unique women's teams in submission: 363


In [4]:
# Load both submission files
submission_stage1 = pd.read_csv(BASE + 'SampleSubmissionStage1.csv')
submission_stage2 = pd.read_csv(BASE + 'SampleSubmissionStage2.csv')

print(f"Stage 1 (practice) rows:  {len(submission_stage1):,}")
print(f"Stage 2 (final) rows:     {len(submission_stage2):,}")

# Check which seasons are in each
submission_stage1['Season'] = submission_stage1['ID'].apply(lambda x: int(x.split('_')[0]))
submission_stage2['Season'] = submission_stage2['ID'].apply(lambda x: int(x.split('_')[0]))

print(f"\nStage 1 seasons: {sorted(submission_stage1['Season'].unique())}")
print(f"Stage 2 seasons: {sorted(submission_stage2['Season'].unique())}")

Stage 1 (practice) rows:  519,144
Stage 2 (final) rows:     132,133

Stage 1 seasons: [np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]
Stage 2 seasons: [np.int64(2026)]


## feature Engineering

In [5]:
def compute_season_stats(results_df):
    """
    Takes game results and computes per-team per-season stats:
    - wins, losses, win%
    - average score, average score allowed
    - average point differential
    """
    
    # --- WINS ---
    # Each row in results_df is one game
    # WTeamID = winner, LTeamID = loser
    wins = results_df.groupby(['Season', 'WTeamID']).size().reset_index()
    wins.columns = ['Season', 'TeamID', 'Wins']
    
    # --- LOSSES ---
    losses = results_df.groupby(['Season', 'LTeamID']).size().reset_index()
    losses.columns = ['Season', 'TeamID', 'Losses']
    
    # --- POINTS SCORED (by winner) ---
    pts_scored_w = results_df.groupby(['Season', 'WTeamID'])['WScore'].mean().reset_index()
    pts_scored_w.columns = ['Season', 'TeamID', 'AvgPtsScored']
    
    # --- POINTS ALLOWED (by winner) ---
    pts_allowed_w = results_df.groupby(['Season', 'WTeamID'])['LScore'].mean().reset_index()
    pts_allowed_w.columns = ['Season', 'TeamID', 'AvgPtsAllowed']

    # --- POINTS SCORED (by loser) ---
    pts_scored_l = results_df.groupby(['Season', 'LTeamID'])['LScore'].mean().reset_index()
    pts_scored_l.columns = ['Season', 'TeamID', 'AvgPtsScored']
    
    # --- POINTS ALLOWED (by loser) ---
    pts_allowed_l = results_df.groupby(['Season', 'LTeamID'])['WScore'].mean().reset_index()
    pts_allowed_l.columns = ['Season', 'TeamID', 'AvgPtsAllowed']
    
    # --- COMBINE WINNER AND LOSER STATS ---
    pts_scored  = pd.concat([pts_scored_w,  pts_scored_l]).groupby(['Season','TeamID'])['AvgPtsScored'].mean().reset_index()
    pts_allowed = pd.concat([pts_allowed_w, pts_allowed_l]).groupby(['Season','TeamID'])['AvgPtsAllowed'].mean().reset_index()
    
    # --- MERGE EVERYTHING TOGETHER ---
    stats = wins.merge(losses, on=['Season','TeamID'], how='outer').fillna(0)
    stats = stats.merge(pts_scored,  on=['Season','TeamID'], how='left')
    stats = stats.merge(pts_allowed, on=['Season','TeamID'], how='left')
    
    # --- CALCULATE DERIVED STATS ---
    stats['GamesPlayed'] = stats['Wins'] + stats['Losses']
    stats['WinPct']      = stats['Wins'] / stats['GamesPlayed']
    stats['PointDiff']   = stats['AvgPtsScored'] - stats['AvgPtsAllowed']
    
    return stats

# Compute for both men and women
print("Computing men's season stats...")
m_stats = compute_season_stats(m_results)

print("Computing women's season stats...")
w_stats = compute_season_stats(w_results)

print()
print("=" * 50)
print("MEN'S SEASON STATS - Sample")
print("=" * 50)
print(m_stats[m_stats['Season'] == 2026].sort_values('WinPct', ascending=False).head(10))

print()
print("=" * 50)
print("WOMEN'S SEASON STATS - Sample")
print("=" * 50)
print(w_stats[w_stats['Season'] == 2026].sort_values('WinPct', ascending=False).head(10))



Computing men's season stats...
Computing women's season stats...

MEN'S SEASON STATS - Sample
       Season  TeamID  Wins  Losses  AvgPtsScored  AvgPtsAllowed  GamesPlayed  \
13554    2026    1275  28.0     1.0     85.178571      81.482143         29.0   
13398    2026    1112  32.0     2.0     81.625000      74.046875         34.0   
13462    2026    1181  32.0     2.0     78.640625      69.406250         34.0   
13555    2026    1276  31.0     3.0     81.182796      73.930108         34.0   
13492    2026    1211  30.0     3.0     76.816667      75.000000         33.0   
13498    2026    1219  27.0     4.0     82.745370      79.023148         31.0   
13444    2026    1163  29.0     5.0     73.320690      69.868966         34.0   
13711    2026    1438  29.0     5.0     76.986207      74.117241         34.0   
13661    2026    1387  27.0     5.0     78.988889      73.811111         32.0   
13390    2026    1103  27.0     5.0     85.296296      81.955556         32.0   

         WinP

## feature Engineering

### Feature 1- win percentage

In [6]:
print("=" * 50)
print("MEN'S TOP 10 TEAMS 2026 - With Names")
print("=" * 50)

# Get top 10 men's teams by win%
m_top = m_stats[m_stats['Season'] == 2026].sort_values('WinPct', ascending=False).head(10)

# Merge with team names
m_top = m_top.merge(m_teams[['TeamID', 'TeamName']], on='TeamID', how='left')

print(m_top[['TeamName', 'Wins', 'Losses', 'WinPct', 'PointDiff']].to_string(index=False))

print()
print("=" * 50)
print("WOMEN'S TOP 10 TEAMS 2026 - With Names")
print("=" * 50)

w_top = w_stats[w_stats['Season'] == 2026].sort_values('WinPct', ascending=False).head(10)
w_top = w_top.merge(w_teams[['TeamID', 'TeamName']], on='TeamID', how='left')

print(w_top[['TeamName', 'Wins', 'Losses', 'WinPct', 'PointDiff']].to_string(index=False))

MEN'S TOP 10 TEAMS 2026 - With Names
   TeamName  Wins  Losses   WinPct  PointDiff
   Miami OH  28.0     1.0 0.965517   3.696429
    Arizona  32.0     2.0 0.941176   7.578125
       Duke  32.0     2.0 0.941176   9.234375
   Michigan  31.0     3.0 0.911765   7.252688
    Gonzaga  30.0     3.0 0.909091   1.816667
 High Point  27.0     4.0 0.870968   3.722222
Connecticut  29.0     5.0 0.852941   3.451724
   Virginia  29.0     5.0 0.852941   2.868966
   St Louis  27.0     5.0 0.843750   5.177778
      Akron  27.0     5.0 0.843750   3.340741

WOMEN'S TOP 10 TEAMS 2026 - With Names
      TeamName  Wins  Losses   WinPct  PointDiff
   Connecticut  34.0     0.0 1.000000  38.382353
          UCLA  31.0     1.0 0.968750   9.193548
South Carolina  31.0     3.0 0.911765  10.881720
         Texas  31.0     3.0 0.911765  11.919355
     Murray St  30.0     3.0 0.909091  -3.483333
     Princeton  26.0     3.0 0.896552   1.673077
   F Dickinson  28.0     4.0 0.875000  -0.160714
     Fairfield  28.0     

### Feature 2- Elo feature

In [7]:
def compute_elo(results_df, k=20, initial_elo=1500):
    """
    Computes Elo ratings for every team after every game.
    
    How Elo works:
    - Every team starts at 1500
    - Win vs strong team  → big Elo gain
    - Win vs weak team    → small Elo gain  
    - Lose vs strong team → small Elo loss
    - Lose vs weak team   → big Elo loss
    - K=20 controls how much each game changes the rating
    """
    
    # Dictionary to store current elo for each team
    elo_dict = {}
    
    # List to store elo AFTER each season ends (for tournament predictions)
    season_end_elo = []
    
    # Sort games by season and day so we process them in order
    results_df = results_df.sort_values(['Season', 'DayNum'])
    
    current_season = None
    
    for _, row in results_df.iterrows():
        season   = row['Season']
        winner   = row['WTeamID']
        loser    = row['LTeamID']
        
        # If new season starts, reset elos slightly toward mean (reversion)
        if season != current_season:
            current_season = season
            for team in elo_dict:
                # Pull rating 25% back toward 1500 each new season
                elo_dict[team] = 0.75 * elo_dict[team] + 0.25 * initial_elo
        
        # Initialize teams if first time we see them
        if winner not in elo_dict:
            elo_dict[winner] = initial_elo
        if loser not in elo_dict:
            elo_dict[loser] = initial_elo
        
        # Get current ratings
        elo_w = elo_dict[winner]
        elo_l = elo_dict[loser]
        
        # Expected win probability based on current ratings
        # This is the core Elo formula
        expected_w = 1 / (1 + 10 ** ((elo_l - elo_w) / 400))
        expected_l = 1 - expected_w
        
        # Update ratings
        # Winner gets points, loser loses points
        elo_dict[winner] = elo_w + k * (1 - expected_w)
        elo_dict[loser]  = elo_l + k * (0 - expected_l)
        
    # After processing all games, save final elo for each team
    for team, elo in elo_dict.items():
        season_end_elo.append({
            'TeamID': team,
            'Elo': elo
        })
    
    return pd.DataFrame(season_end_elo)

# Compute Elo for men and women
print("Computing men's Elo ratings...")
m_elo = compute_elo(m_results)

print("Computing women's Elo ratings...")
w_elo = compute_elo(w_results)

# Merge with team names to see results
m_elo_named = m_elo.merge(m_teams[['TeamID','TeamName']], on='TeamID')
w_elo_named = w_elo.merge(w_teams[['TeamID','TeamName']], on='TeamID')

print()
print("=" * 50)
print("MEN'S TOP 15 BY ELO - 2026")
print("=" * 50)
print(m_elo_named.sort_values('Elo', ascending=False).head(15)[['TeamName','Elo']].to_string(index=False))

print()
print("=" * 50)
print("WOMEN'S TOP 15 BY ELO - 2026")
print("=" * 50)
print(w_elo_named.sort_values('Elo', ascending=False).head(15)[['TeamName','Elo']].to_string(index=False))




Computing men's Elo ratings...
Computing women's Elo ratings...

MEN'S TOP 15 BY ELO - 2026
    TeamName         Elo
        Duke 1871.466544
     Arizona 1849.083499
     Houston 1825.324333
    Michigan 1820.119284
   St John's 1803.600830
     Gonzaga 1800.001362
     Florida 1798.085198
 Connecticut 1779.554457
St Mary's CA 1770.149741
      Purdue 1766.919161
 Michigan St 1758.797583
     Iowa St 1753.566555
     Utah St 1753.503154
     Alabama 1753.014114
         VCU 1750.175986

WOMEN'S TOP 15 BY ELO - 2026
      TeamName         Elo
   Connecticut 1926.650590
          UCLA 1923.102032
South Carolina 1920.521892
         Texas 1906.951907
           LSU 1820.263967
          Iowa 1810.436964
          Duke 1807.351173
           TCU 1801.911147
       Ohio St 1798.435954
 West Virginia 1785.716412
North Carolina 1781.513478
    Vanderbilt 1775.900749
      Michigan 1773.153863
      Oklahoma 1772.524475
    Louisville 1771.560423


### Feature 3 — Seeds

In [8]:
# Clean seed numbers — remove region letter and play-in letters
# Example: "W01" → 1,  "X16a" → 16

def extract_seed_number(seed_str):
    """
    Converts seed string to integer
    'W01' → 1
    'X12' → 12  
    'Z16a' → 16
    """
    # Remove first letter (region) and any trailing a/b (play-in)
    cleaned = seed_str[1:]        # Remove region letter: 'W01' → '01'
    cleaned = cleaned.replace('a','').replace('b','')  # Remove play-in: '16a' → '16'
    return int(cleaned)

# Apply to men's and women's seeds
m_seeds['SeedNum'] = m_seeds['Seed'].apply(extract_seed_number)
w_seeds['SeedNum'] = w_seeds['Seed'].apply(extract_seed_number)

# Look at 2026 seeds
print("=" * 50)
print("MEN'S 2026 TOURNAMENT SEEDS")
print("=" * 50)
m_seeds_2026 = m_seeds[m_seeds['Season']==2026].merge(
    m_teams[['TeamID','TeamName']], on='TeamID'
)
print(m_seeds_2026[['Seed','SeedNum','TeamName']].sort_values('SeedNum').head(20).to_string(index=False))

print()
print("=" * 50)
print("WOMEN'S 2026 TOURNAMENT SEEDS")
print("=" * 50)
w_seeds_2026 = w_seeds[w_seeds['Season']==2026].merge(
    w_teams[['TeamID','TeamName']], on='TeamID'
)
print(w_seeds_2026[['Seed','SeedNum','TeamName']].sort_values('SeedNum').head(20).to_string(index=False))

print()
print("=" * 50)
print("HISTORICAL SEED WIN RATES")
print("=" * 50)

# Merge seeds into tournament results to see how seeds perform historically
m_tourney_seeds = m_tourney.merge(
    m_seeds[['Season','TeamID','SeedNum']], 
    left_on=['Season','WTeamID'], right_on=['Season','TeamID']
).rename(columns={'SeedNum':'WinnerSeed'}).drop('TeamID', axis=1)

m_tourney_seeds = m_tourney_seeds.merge(
    m_seeds[['Season','TeamID','SeedNum']], 
    left_on=['Season','LTeamID'], right_on=['Season','TeamID']
).rename(columns={'SeedNum':'LoserSeed'}).drop('TeamID', axis=1)

# What % of time does the lower seed number (better team) win?
m_tourney_seeds['UpsetOccurred'] = m_tourney_seeds['WinnerSeed'] > m_tourney_seeds['LoserSeed']

print(f"Overall upset rate in men's tournament: {m_tourney_seeds['UpsetOccurred'].mean():.1%}")
print()

# Win rate by seed matchup — classic 1v16, 2v15 etc
print("Seed 1 win rate vs Seed 16:", 
      f"{(1 - m_tourney_seeds[m_tourney_seeds['LoserSeed']==16]['UpsetOccurred'].mean()):.1%}")
print("Seed 2 win rate vs Seed 15:", 
      f"{(1 - m_tourney_seeds[m_tourney_seeds['LoserSeed']==15]['UpsetOccurred'].mean()):.1%}")
print("Seed 5 win rate vs Seed 12:", 
      f"{(1 - m_tourney_seeds[m_tourney_seeds['LoserSeed']==12]['UpsetOccurred'].mean()):.1%}")
print("Seed 8 win rate vs Seed 9:", 
      f"{(1 - m_tourney_seeds[m_tourney_seeds['LoserSeed']==9]['UpsetOccurred'].mean()):.1%}")


MEN'S 2026 TOURNAMENT SEEDS
Seed  SeedNum    TeamName
 W01        1        Duke
 X01        1     Florida
 Z01        1     Arizona
 Y01        1    Michigan
 Y02        2     Iowa St
 Z02        2      Purdue
 X02        2     Houston
 W02        2 Connecticut
 W03        3 Michigan St
 X03        3    Illinois
 Z03        3     Gonzaga
 Y03        3    Virginia
 Y04        4     Alabama
 Z04        4    Arkansas
 X04        4    Nebraska
 W04        4      Kansas
 W05        5   St John's
 X05        5  Vanderbilt
 Z05        5   Wisconsin
 Y05        5  Texas Tech

WOMEN'S 2026 TOURNAMENT SEEDS
Seed  SeedNum       TeamName
 W01        1    Connecticut
 X01        1 South Carolina
 Z01        1           UCLA
 Y01        1          Texas
 Y02        2       Michigan
 Z02        2            LSU
 X02        2           Iowa
 W02        2     Vanderbilt
 W03        3        Ohio St
 X03        3            TCU
 Z03        3           Duke
 Y03        3     Louisville
 Y04        4  Wes

In [9]:
print("=" * 50)
print("HISTORICAL SEED WIN RATES - FIXED")
print("=" * 50)

def seed_matchup_winrate(tourney_seeds_df, seed_a, seed_b):
    """
    Returns win rate of seed_a when playing against seed_b
    seed_a is assumed to be the better seed (lower number)
    """
    # Get all games between these two seeds
    matchups = tourney_seeds_df[
        ((tourney_seeds_df['WinnerSeed'] == seed_a) & 
         (tourney_seeds_df['LoserSeed']  == seed_b)) |
        ((tourney_seeds_df['WinnerSeed'] == seed_b) & 
         (tourney_seeds_df['LoserSeed']  == seed_a))
    ]
    
    if len(matchups) == 0:
        return None, 0
    
    # Seed_a wins when it appears as WinnerSeed
    seed_a_wins = (matchups['WinnerSeed'] == seed_a).sum()
    total_games  = len(matchups)
    win_rate     = seed_a_wins / total_games
    
    return win_rate, total_games

print(f"Overall upset rate: {m_tourney_seeds['UpsetOccurred'].mean():.1%}")
print()
print(f"{'Matchup':<12} {'Seed1 WinRate':>14} {'Games Played':>13}")
print("-" * 42)

matchups_to_check = [
    (1, 16), (2, 15), (3, 14), (4, 13),
    (5, 12), (6, 11), (7, 10), (8,  9)
]

for seed_a, seed_b in matchups_to_check:
    rate, n = seed_matchup_winrate(m_tourney_seeds, seed_a, seed_b)
    if rate is not None:
        print(f"  {seed_a} vs {seed_b:<8} {rate:>13.1%} {n:>12} games")

print()
print("=" * 50)
print("KEY INSIGHT — Famous Upset Matchups")
print("=" * 50)
print("5 vs 12 is famous for upsets in March Madness")
print("8 vs 9  is basically a coin flip")
print("1 vs 16 almost never upsets (until 2018!)")


HISTORICAL SEED WIN RATES - FIXED
Overall upset rate: 27.3%

Matchup       Seed1 WinRate  Games Played
------------------------------------------
  1 vs 16               98.8%          160 games
  2 vs 15               93.1%          160 games
  3 vs 14               85.6%          160 games
  4 vs 13               79.4%          160 games
  5 vs 12               64.4%          160 games
  6 vs 11               61.3%          160 games
  7 vs 10               61.0%          159 games
  8 vs 9                48.1%          160 games

KEY INSIGHT — Famous Upset Matchups
5 vs 12 is famous for upsets in March Madness
8 vs 9  is basically a coin flip
1 vs 16 almost never upsets (until 2018!)


### build training data, combine features

In [10]:
def build_training_data(tourney_df, stats_df, elo_df, seeds_df):
    """
    For every historical tournament game, build one row with:
    - Feature differences between the two teams
    - Result: 1 if lower TeamID won, 0 if higher TeamID won
    
    We always put lower TeamID as Team1 to match submission format
    """
    
    rows = []
    
    for _, game in tourney_df.iterrows():
        season  = game['Season']
        wteam   = game['WTeamID']   # winner
        lteam   = game['LTeamID']   # loser
        
        # Always assign lower ID as team1 (matches submission format)
        if wteam < lteam:
            team1, team2 = wteam, lteam
            result = 1   # team1 (lower ID) won
        else:
            team1, team2 = lteam, wteam
            result = 0   # team1 (lower ID) lost
        
        # --- LOOK UP STATS FOR BOTH TEAMS ---
        s1 = stats_df[(stats_df['Season']==season) & 
                      (stats_df['TeamID']==team1)]
        s2 = stats_df[(stats_df['Season']==season) & 
                      (stats_df['TeamID']==team2)]
        
        # --- LOOK UP ELO FOR BOTH TEAMS ---
        e1 = elo_df[elo_df['TeamID']==team1]
        e2 = elo_df[elo_df['TeamID']==team2]
        
        # --- LOOK UP SEEDS FOR BOTH TEAMS ---
        seed1 = seeds_df[(seeds_df['Season']==season) & 
                         (seeds_df['TeamID']==team1)]
        seed2 = seeds_df[(seeds_df['Season']==season) & 
                         (seeds_df['TeamID']==team2)]
        
        # Skip if any data is missing
        if any(len(x)==0 for x in [s1, s2, e1, e2, seed1, seed2]):
            continue
        
        rows.append({
            'Season'      : season,
            'Team1'       : team1,
            'Team2'       : team2,
            # Differences (Team1 minus Team2)
            # Positive = Team1 is better
            'EloDiff'     : e1['Elo'].values[0]        - e2['Elo'].values[0],
            'WinPctDiff'  : s1['WinPct'].values[0]     - s2['WinPct'].values[0],
            'PointDiff'   : s1['PointDiff'].values[0]  - s2['PointDiff'].values[0],
            'SeedDiff'    : seed2['SeedNum'].values[0] - seed1['SeedNum'].values[0],
            # Note: SeedDiff is seed2 - seed1 because lower seed number = better team
            # So positive SeedDiff means Team1 has better seed
            'Result'      : result
        })
    
    return pd.DataFrame(rows)

# Build training data for men and women
print("Building men's training data...")
m_train = build_training_data(m_tourney, m_stats, m_elo, m_seeds)

print("Building women's training data...")
w_train = build_training_data(w_tourney, w_stats, w_elo, w_seeds)

print()
print("=" * 50)
print("TRAINING DATA SUMMARY")
print("=" * 50)
print(f"Men's training rows:   {len(m_train):,}")
print(f"Women's training rows: {len(w_train):,}")
print()
print("Sample men's training data:")
print(m_train.head(10).to_string(index=False))
print()
print("Feature correlations with Result (men's):")
print(m_train[['EloDiff','WinPctDiff','PointDiff','SeedDiff','Result']].corr()['Result'].sort_values(ascending=False))


Building men's training data...
Building women's training data...

TRAINING DATA SUMMARY
Men's training rows:   2,585
Women's training rows: 1,717

Sample men's training data:
 Season  Team1  Team2     EloDiff  WinPctDiff  PointDiff  SeedDiff  Result
   1985   1116   1234  109.015874   -0.030303  -5.350000        -1       1
   1985   1120   1345 -107.238214   -0.059310   2.208779        -5       1
   1985   1207   1250   77.635397    0.546616  10.186667        15       1
   1985   1229   1425    2.069117    0.062169   0.305744        -1       1
   1985   1242   1325  250.419326    0.025926  -0.021118        11       1
   1985   1246   1449  138.872906   -0.138249  -0.838384        -7       1
   1985   1256   1338   18.430801    0.351648   1.529720         7       1
   1985   1233   1260   20.306781    0.028736   1.300000        -9       0
   1985   1292   1314 -188.661001   -0.241935  -3.161789       -13       0
   1985   1323   1333  -53.187169   -0.009852   2.957738         3       1

In [11]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import brier_score_loss
import warnings
warnings.filterwarnings('ignore')

# ============================================
# FEATURES WE USE TO TRAIN
# ============================================
FEATURES = ['EloDiff', 'WinPctDiff', 'PointDiff', 'SeedDiff']

def train_model(train_df, gender='Men'):
    """
    Trains a logistic regression model on historical tournament data
    Returns trained model and scaler
    """
    print(f"Training {gender}'s model...")
    
    X = train_df[FEATURES].values
    y = train_df['Result'].values
    
    # --- SCALE FEATURES ---
    # Logistic regression works better when features are on same scale
    # EloDiff ranges -500 to +500
    # WinPctDiff ranges -1 to +1
    # We normalize everything to similar scale
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    
    # --- TRAIN MODEL ---
    # Logistic regression outputs probabilities between 0 and 1
    # Perfect for our use case!
    model = LogisticRegression(C=1.0, max_iter=1000)
    model.fit(X_scaled, y)
    
    # --- CHECK FEATURE IMPORTANCE ---
    print(f"\n{gender}'s Feature Coefficients:")
    for feat, coef in zip(FEATURES, model.coef_[0]):
        print(f"  {feat:<15} {coef:+.4f}")
    
    # --- EVALUATE ON TRAINING DATA ---
    # Note: in a real project you'd use cross-validation
    # We'll do that next
    y_pred = model.predict_proba(X_scaled)[:, 1]
    brier  = brier_score_loss(y, y_pred)
    print(f"\n{gender}'s Training Brier Score: {brier:.4f}")
    print(f"  (Random guessing = 0.25, lower is better)")
    
    return model, scaler

# Train both models
m_model, m_scaler = train_model(m_train, 'Men')
print()
w_model, w_scaler = train_model(w_train, 'Women')

print()
print("=" * 50)
print("QUICK SANITY CHECK")
print("=" * 50)
print("What probability does our model give Duke (#1 seed)")
print("vs a #16 seed?")

# Duke is seed 1, opponent is seed 16
# Seed diff = 16 - 1 = 15 (very positive, team1 is much better)
test_matchup = pd.DataFrame([{
    'EloDiff'    :  300,   # Duke has much higher Elo
    'WinPctDiff' :  0.3,   # Duke wins 30% more games
    'PointDiff'  :  8.0,   # Duke wins by 8 more points
    'SeedDiff'   :  15     # Duke is seed 1, opponent seed 16
}])

test_scaled = m_scaler.transform(test_matchup[FEATURES])
prob = m_model.predict_proba(test_scaled)[0][1]
print(f"\nModel predicts: {prob:.1%} chance Duke wins")
print("Historical reality: 98.8%")


Training Men's model...

Men's Feature Coefficients:
  EloDiff         +0.1760
  WinPctDiff      +0.0833
  PointDiff       +0.3201
  SeedDiff        +0.9584

Men's Training Brier Score: 0.1867
  (Random guessing = 0.25, lower is better)

Training Women's model...

Women's Feature Coefficients:
  EloDiff         +0.1395
  WinPctDiff      +0.0577
  PointDiff       +0.6982
  SeedDiff        +1.6909

Women's Training Brier Score: 0.1421
  (Random guessing = 0.25, lower is better)

QUICK SANITY CHECK
What probability does our model give Duke (#1 seed)
vs a #16 seed?

Model predicts: 95.1% chance Duke wins
Historical reality: 98.8%


In [12]:
from sklearn.metrics import brier_score_loss

def cross_validate_by_season(train_df, gender='Men'):
    """
    Train on all seasons EXCEPT the last one
    Test on the last season
    Repeat for each season — this is called walk-forward validation
    It simulates real prediction: you only use PAST data to predict FUTURE
    """
    
    seasons  = sorted(train_df['Season'].unique())
    
    # Need at least 5 seasons to train before testing
    test_seasons  = seasons[5:]
    
    brier_scores = []
    
    for test_season in test_seasons:
        # Train on everything BEFORE this season
        train_mask = train_df['Season'] < test_season
        test_mask  = train_df['Season'] == test_season
        
        train_data = train_df[train_mask]
        test_data  = train_df[test_mask]
        
        if len(test_data) == 0:
            continue
        
        # Train model
        X_train = train_data[FEATURES].values
        y_train = train_data['Result'].values
        X_test  = test_data[FEATURES].values
        y_test  = test_data['Result'].values
        
        scaler  = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_test_scaled  = scaler.transform(X_test)
        
        model   = LogisticRegression(C=1.0, max_iter=1000)
        model.fit(X_train_scaled, y_train)
        
        # Predict on test season
        y_pred  = model.predict_proba(X_test_scaled)[:, 1]
        brier   = brier_score_loss(y_test, y_pred)
        brier_scores.append({'Season': test_season, 'Brier': brier})
    
    results_df = pd.DataFrame(brier_scores)
    
    print(f"\n{gender}'s Walk-Forward Validation Results:")
    print(f"{'Season':<10} {'Brier Score':>12}")
    print("-" * 25)
    for _, row in results_df.iterrows():
        print(f"  {int(row['Season']):<8} {row['Brier']:>12.4f}")
    
    print(f"\n  Average Brier Score: {results_df['Brier'].mean():.4f}")
    print(f"  Best Season:         {results_df.loc[results_df['Brier'].idxmin(), 'Season']:.0f}  ({results_df['Brier'].min():.4f})")
    print(f"  Worst Season:        {results_df.loc[results_df['Brier'].idxmax(), 'Season']:.0f}  ({results_df['Brier'].max():.4f})")
    
    return results_df

# Run cross validation for both
m_cv = cross_validate_by_season(m_train, 'Men')
print()
w_cv = cross_validate_by_season(w_train, 'Women')


Men's Walk-Forward Validation Results:
Season      Brier Score
-------------------------
  1990           0.1900
  1991           0.1839
  1992           0.1715
  1993           0.1672
  1994           0.1721
  1995           0.1776
  1996           0.1754
  1997           0.1775
  1998           0.1993
  1999           0.1937
  2000           0.1804
  2001           0.1903
  2002           0.1955
  2003           0.1857
  2004           0.1692
  2005           0.1790
  2006           0.1976
  2007           0.1439
  2008           0.1653
  2009           0.1686
  2010           0.1962
  2011           0.2056
  2012           0.1895
  2013           0.2143
  2014           0.2132
  2015           0.1685
  2016           0.1955
  2017           0.1812
  2018           0.2162
  2019           0.1736
  2021           0.2377
  2022           0.2265
  2023           0.2037
  2024           0.1958
  2025           0.1534

  Average Brier Score: 0.1873
  Best Season:         2007  (0.1439)
 

In [13]:
def generate_submission(submission_df, 
                        m_model, m_scaler, m_stats, m_elo, m_seeds,
                        w_model, w_scaler, w_stats, w_elo, w_seeds):
    """
    For every row in submission file:
    1. Parse Team1 and Team2 from the ID
    2. Look up their features
    3. Predict probability
    4. Fill in the Pred column
    """
    
    predictions = []
    missing     = 0
    
    for _, row in submission_df.iterrows():
        # Parse the ID
        parts   = row['ID'].split('_')
        season  = int(parts[0])
        team1   = int(parts[1])
        team2   = int(parts[2])
        
        # Determine men's or women's based on team ID range
        is_mens = team1 < 2000
        
        if is_mens:
            stats_df = m_stats
            elo_df   = m_elo
            seeds_df = m_seeds
            model    = m_model
            scaler   = m_scaler
        else:
            stats_df = w_stats
            elo_df   = w_elo
            seeds_df = w_seeds
            model    = w_model
            scaler   = w_scaler
        
        # Look up stats
        s1 = stats_df[(stats_df['Season']==season) & (stats_df['TeamID']==team1)]
        s2 = stats_df[(stats_df['Season']==season) & (stats_df['TeamID']==team2)]
        e1 = elo_df[elo_df['TeamID']==team1]
        e2 = elo_df[elo_df['TeamID']==team2]
        
        # Seeds are optional — not all teams make tournament
        seed1 = seeds_df[(seeds_df['Season']==season) & (seeds_df['TeamID']==team1)]
        seed2 = seeds_df[(seeds_df['Season']==season) & (seeds_df['TeamID']==team2)]
        
        # Handle missing stats gracefully
        if any(len(x)==0 for x in [s1, s2, e1, e2]):
            predictions.append(0.5)
            missing += 1
            continue
        
        # For teams without seeds, use neutral value of 8.5 (middle of 1-16)
        s1_num = seed1['SeedNum'].values[0] if len(seed1) > 0 else 8.5
        s2_num = seed2['SeedNum'].values[0] if len(seed2) > 0 else 8.5
        
        # Build feature vector
        features = pd.DataFrame([{
            'EloDiff'    : e1['Elo'].values[0]       - e2['Elo'].values[0],
            'WinPctDiff' : s1['WinPct'].values[0]    - s2['WinPct'].values[0],
            'PointDiff'  : s1['PointDiff'].values[0] - s2['PointDiff'].values[0],
            'SeedDiff'   : s2_num                    - s1_num
        }])
        
        # Scale and predict
        features_scaled = scaler.transform(features[FEATURES])
        prob = model.predict_proba(features_scaled)[0][1]
        
        # Clip to avoid extreme values (good for Brier score)
        prob = np.clip(prob, 0.025, 0.975)
        
        predictions.append(prob)
    
    submission_df = submission_df.copy()
    submission_df['Pred'] = predictions
    
    print(f"✅ Predictions generated!")
    print(f"   Total rows:       {len(predictions):,}")
    print(f"   Missing/defaulted: {missing:,}")
    print(f"   Min prediction:   {min(predictions):.4f}")
    print(f"   Max prediction:   {max(predictions):.4f}")
    print(f"   Mean prediction:  {sum(predictions)/len(predictions):.4f}")
    
    return submission_df

# Generate predictions
print("Generating predictions for all 2026 matchups...")
final_submission = generate_submission(
    submission_stage2,
    m_model, m_scaler, m_stats, m_elo, m_seeds,
    w_model, w_scaler, w_stats, w_elo, w_seeds
)

print()
print("=" * 50)
print("FINAL SUBMISSION PREVIEW")
print("=" * 50)
print(final_submission.head(10).to_string(index=False))

# Save to file
final_submission[['ID','Pred']].to_csv('submission.csv', index=False)
print()
print("✅ submission.csv saved!")
print("   Ready to upload to Kaggle!")

Generating predictions for all 2026 matchups...
✅ Predictions generated!
   Total rows:       132,133
   Missing/defaulted: 0
   Min prediction:   0.0250
   Max prediction:   0.9750
   Mean prediction:  0.5168

FINAL SUBMISSION PREVIEW
            ID     Pred  Season
2026_1101_1102 0.620688    2026
2026_1101_1103 0.349932    2026
2026_1101_1104 0.173840    2026
2026_1101_1105 0.478224    2026
2026_1101_1106 0.458782    2026
2026_1101_1107 0.457285    2026
2026_1101_1108 0.610533    2026
2026_1101_1110 0.415777    2026
2026_1101_1111 0.380041    2026
2026_1101_1112 0.073646    2026

✅ submission.csv saved!
   Ready to upload to Kaggle!


In [14]:
# Load Stage 1 submission file
submission_stage1 = pd.read_csv(BASE + 'SampleSubmissionStage1.csv')

print("Stage 1 file info:")
print(f"  Total rows: {len(submission_stage1):,}")

# Check which seasons are in it
submission_stage1['Season'] = submission_stage1['ID'].apply(lambda x: int(x.split('_')[0]))
print(f"  Seasons: {sorted(submission_stage1['Season'].unique())}")

# Generate predictions using same function we already built
print()
print("Generating predictions for 2022-2025 matchups...")
stage1_submission = generate_submission(
    submission_stage1,
    m_model, m_scaler, m_stats, m_elo, m_seeds,
    w_model, w_scaler, w_stats, w_elo, w_seeds
)

print()
print("=" * 50)
print("STAGE 1 SUBMISSION PREVIEW")
print("=" * 50)
print(stage1_submission[['ID','Pred']].head(10).to_string(index=False))

# Save it
stage1_submission[['ID','Pred']].to_csv('submission_stage1.csv', index=False)
print()
print("✅ submission_stage1.csv saved!")


Stage 1 file info:
  Total rows: 519,144
  Seasons: [np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]

Generating predictions for 2022-2025 matchups...
✅ Predictions generated!
   Total rows:       519,144
   Missing/defaulted: 0
   Min prediction:   0.0250
   Max prediction:   0.9750
   Mean prediction:  0.5096

STAGE 1 SUBMISSION PREVIEW
            ID     Pred
2022_1101_1102 0.671249
2022_1101_1103 0.560561
2022_1101_1104 0.343130
2022_1101_1105 0.631656
2022_1101_1106 0.636548
2022_1101_1107 0.613487
2022_1101_1108 0.652315
2022_1101_1110 0.617550
2022_1101_1111 0.524629
2022_1101_1112 0.151674

✅ submission_stage1.csv saved!


In [15]:
print("=" * 50)
print("MASSEY ORDINALS - Overview")
print("=" * 50)
print(m_massey.head(10).to_string(index=False))
print()
print(f"Total rows: {len(m_massey):,}")
print(f"Seasons:    {m_massey['Season'].min()} → {m_massey['Season'].max()}")
print(f"Unique ranking systems: {m_massey['SystemName'].nunique()}")
print()

# Show all available systems
print("All ranking systems available:")
print(sorted(m_massey['SystemName'].unique()))
print()

# How many systems are available for 2026?
massey_2026 = m_massey[m_massey['Season'] == 2026]
print(f"Systems available for 2026: {massey_2026['SystemName'].nunique()}")
print(f"Latest RankingDayNum in 2026: {massey_2026['RankingDayNum'].max()}")
print()

# Show the key systems we care about
key_systems = ['POM', 'SAG', 'RPI', 'AP', 'ESPN', 'MOR', 'KPI', 'NET']
print("Key systems available in 2026:")
for sys in key_systems:
    available = sys in massey_2026['SystemName'].values
    count     = len(massey_2026[massey_2026['SystemName']==sys])
    print(f"  {sys:<6} {'✅' if available else '❌'}  ({count} teams ranked)")

MASSEY ORDINALS - Overview
 Season  RankingDayNum SystemName  TeamID  OrdinalRank
   2003             35        SEL    1102          159
   2003             35        SEL    1103          229
   2003             35        SEL    1104           12
   2003             35        SEL    1105          314
   2003             35        SEL    1106          260
   2003             35        SEL    1107          249
   2003             35        SEL    1108          228
   2003             35        SEL    1110          204
   2003             35        SEL    1111          183
   2003             35        SEL    1112           26

Total rows: 5,865,001
Seasons:    2003 → 2026
Unique ranking systems: 197

All ranking systems available:
['7OT', 'ACU', 'ADE', 'AEI', 'AP', 'ARG', 'ATP', 'AUS', 'AWS', 'BAR', 'BBT', 'BCM', 'BD', 'BIH', 'BKM', 'BLS', 'BMN', 'BNM', 'BNT', 'BNZ', 'BOB', 'BOW', 'BP5', 'BPI', 'BRZ', 'BUR', 'BWE', 'CBR', 'CJB', 'CMV', 'CNG', 'COL', 'COX', 'CPA', 'CPR', 'CRO', 'CRW', 'CT

In [16]:
print("=" * 50)
print("MASSEY ORDINALS - Overview")
print("=" * 50)
print(m_massey.head(10).to_string(index=False))
print()
print(f"Total rows: {len(m_massey):,}")
print(f"Seasons:    {m_massey['Season'].min()} → {m_massey['Season'].max()}")
print(f"Unique ranking systems: {m_massey['SystemName'].nunique()}")
print()

# Show all available systems
print("All ranking systems available:")
print(sorted(m_massey['SystemName'].unique()))
print()

# How many systems are available for 2026?
massey_2026 = m_massey[m_massey['Season'] == 2026]
print(f"Systems available for 2026: {massey_2026['SystemName'].nunique()}")
print(f"Latest RankingDayNum in 2026: {massey_2026['RankingDayNum'].max()}")
print()

# Show the key systems we care about
key_systems = ['POM', 'SAG', 'RPI', 'AP', 'ESPN', 'MOR', 'KPI', 'NET']
print("Key systems available in 2026:")
for sys in key_systems:
    available = sys in massey_2026['SystemName'].values
    count     = len(massey_2026[massey_2026['SystemName']==sys])
    print(f"  {sys:<6} {'✅' if available else '❌'}  ({count} teams ranked)")

MASSEY ORDINALS - Overview
 Season  RankingDayNum SystemName  TeamID  OrdinalRank
   2003             35        SEL    1102          159
   2003             35        SEL    1103          229
   2003             35        SEL    1104           12
   2003             35        SEL    1105          314
   2003             35        SEL    1106          260
   2003             35        SEL    1107          249
   2003             35        SEL    1108          228
   2003             35        SEL    1110          204
   2003             35        SEL    1111          183
   2003             35        SEL    1112           26

Total rows: 5,865,001
Seasons:    2003 → 2026
Unique ranking systems: 197

All ranking systems available:
['7OT', 'ACU', 'ADE', 'AEI', 'AP', 'ARG', 'ATP', 'AUS', 'AWS', 'BAR', 'BBT', 'BCM', 'BD', 'BIH', 'BKM', 'BLS', 'BMN', 'BNM', 'BNT', 'BNZ', 'BOB', 'BOW', 'BP5', 'BPI', 'BRZ', 'BUR', 'BWE', 'CBR', 'CJB', 'CMV', 'CNG', 'COL', 'COX', 'CPA', 'CPR', 'CRO', 'CRW', 'CT

In [17]:
# ============================================
# STEP 1: Pick only systems confirmed available in 2026
# ============================================
GOOD_SYSTEMS = ['POM', 'RPI', 'MOR']  # removed NET and AP

# Check what's actually available in 2026
available_2026 = m_massey[
    (m_massey['Season'] == 2026) & 
    (m_massey['RankingDayNum'] == 133)
]['SystemName'].unique()

print(f"Systems available in 2026 at day 133: {len(available_2026)}")
print(sorted(available_2026))

# Only keep systems that exist in 2026
final_rankings = m_massey[m_massey['RankingDayNum'] == 133].copy()

print()
print("Coverage of our chosen systems in 2026:")
for sys in GOOD_SYSTEMS:
    count = len(final_rankings[
        (final_rankings['Season']==2026) & 
        (final_rankings['SystemName']==sys)
    ])
    print(f"  {sys}: {count} teams ranked")

# ============================================  
# STEP 2: Create composite ranking
# ============================================
def build_composite_ranking(massey_df, systems):
    filtered = massey_df[
        (massey_df['SystemName'].isin(systems)) &
        (massey_df['RankingDayNum'] == 133)
    ].copy()
    
    composite = filtered.groupby(['Season', 'TeamID'])['OrdinalRank'].mean().reset_index()
    composite.columns = ['Season', 'TeamID', 'CompositeRank']
    return composite

m_composite = build_composite_ranking(m_massey, GOOD_SYSTEMS)

print()
print("=" * 50)
print("COMPOSITE RANKINGS 2026 - Top 20 Men's Teams")
print("=" * 50)
top_ranked = m_composite[m_composite['Season']==2026].sort_values('CompositeRank').head(20)
top_ranked = top_ranked.merge(m_teams[['TeamID','TeamName']], on='TeamID')
print(top_ranked[['TeamName','CompositeRank']].to_string(index=False))

print()
print("=" * 50)
print("ELO vs COMPOSITE RANK - Top 10 Comparison")
print("=" * 50)
elo_top = m_elo.merge(m_teams[['TeamID','TeamName']], on='TeamID')
elo_top = elo_top.sort_values('Elo', ascending=False).head(10).reset_index(drop=True)

rank_top = top_ranked.head(10).reset_index(drop=True)

print(f"{'Elo Rank':<5} {'Team (Elo)':<15} {'Massey Rank':<5} {'Team (Massey)':<15}")
print("-" * 45)
for i in range(10):
    print(f"  {i+1:<6} {elo_top.loc[i,'TeamName']:<15} {i+1:<8} {rank_top.loc[i,'TeamName']:<15}")

Systems available in 2026 at day 133: 19
['BIH', 'BMN', 'DII', 'DOK', 'ENG', 'KPK', 'LMC', 'MAS', 'MB', 'MOR', 'NOL', 'OMN', 'PGH', 'POM', 'RPI', 'TRK', 'WEI', 'WIL', 'WLS']

Coverage of our chosen systems in 2026:
  POM: 365 teams ranked
  RPI: 365 teams ranked
  MOR: 365 teams ranked

COMPOSITE RANKINGS 2026 - Top 20 Men's Teams
   TeamName  CompositeRank
       Duke       1.333333
   Michigan       1.666667
    Arizona       3.000000
    Florida       4.333333
    Houston       5.000000
Michigan St      10.000000
    Iowa St      10.333333
   Virginia      11.000000
    Gonzaga      11.000000
   Illinois      11.666667
     Purdue      11.666667
 Vanderbilt      11.666667
Connecticut      12.000000
  St John's      13.666667
     Kansas      16.000000
   Arkansas      16.666667
    Alabama      16.666667
   Nebraska      17.666667
 Texas Tech      19.333333
 Louisville      20.000000

ELO vs COMPOSITE RANK - Top 10 Comparison
Elo Rank Team (Elo)      Massey Rank Team (Massey)  
----

```

---

## What to Look For
```
Do Elo and Massey agree on top teams?
→ If yes: both are capturing real team strength
→ If different: interesting! Massey uses more info

We expect both to have Duke, Arizona, Houston in top 5

In [18]:
# ============================================
# STEP 1 - Extract Massey Rankings
# ============================================

systems_to_use = ['POM', 'MOR', 'RPI']

def get_massey_features(massey_df, season, systems=systems_to_use):
    """
    Gets pre-tournament rankings for each team in a given season
    Lower rank = better team, we convert so higher score = better
    """
    season_data = massey_df[massey_df['Season'] == season]
    max_day     = season_data['RankingDayNum'].max()
    latest      = season_data[season_data['RankingDayNum'] == max_day]
    
    all_teams_ranks = {}
    
    for system in systems:
        sys_data = latest[latest['SystemName'] == system][['TeamID', 'OrdinalRank']].copy()
        
        if len(sys_data) == 0:
            continue
        
        max_rank = sys_data['OrdinalRank'].max()
        sys_data[f'{system}_Score'] = max_rank - sys_data['OrdinalRank'] + 1
        
        for _, row in sys_data.iterrows():
            tid = row['TeamID']
            if tid not in all_teams_ranks:
                all_teams_ranks[tid] = {}
            all_teams_ranks[tid][f'{system}_Score'] = row[f'{system}_Score']
    
    if not all_teams_ranks:
        return pd.DataFrame(columns=['TeamID', 'CompositeRank'])
    
    ranks_df = pd.DataFrame.from_dict(all_teams_ranks, orient='index')
    ranks_df.index.name = 'TeamID'
    ranks_df = ranks_df.reset_index()
    
    score_cols = [f'{s}_Score' for s in systems if f'{s}_Score' in ranks_df.columns]
    ranks_df['CompositeRank'] = ranks_df[score_cols].mean(axis=1)
    
    return ranks_df

# Build rankings for every season
print("Extracting Massey rankings for all seasons...")
massey_by_season = {}

for season in sorted(m_massey['Season'].unique()):
    massey_by_season[season] = get_massey_features(m_massey, season)

print(f"✅ Rankings extracted for {len(massey_by_season)} seasons")

# ============================================
# STEP 2 - Build Training Data V2
# ============================================

def build_training_data_v2(tourney_df, stats_df, elo_df, seeds_df, massey_by_season):
    rows    = []
    skipped = 0
    
    for _, game in tourney_df.iterrows():
        season = game['Season']
        wteam  = game['WTeamID']
        lteam  = game['LTeamID']
        
        if wteam < lteam:
            team1, team2 = wteam, lteam
            result = 1
        else:
            team1, team2 = lteam, wteam
            result = 0
        
        s1    = stats_df[(stats_df['Season']==season) & (stats_df['TeamID']==team1)]
        s2    = stats_df[(stats_df['Season']==season) & (stats_df['TeamID']==team2)]
        e1    = elo_df[elo_df['TeamID']==team1]
        e2    = elo_df[elo_df['TeamID']==team2]
        seed1 = seeds_df[(seeds_df['Season']==season) & (seeds_df['TeamID']==team1)]
        seed2 = seeds_df[(seeds_df['Season']==season) & (seeds_df['TeamID']==team2)]
        
        if any(len(x)==0 for x in [s1, s2, e1, e2, seed1, seed2]):
            skipped += 1
            continue
        
        # Massey — default 0 if not available for that season
        massey_diff = 0
        if season in massey_by_season:
            m_ranks = massey_by_season[season]
            m1 = m_ranks[m_ranks['TeamID']==team1]
            m2 = m_ranks[m_ranks['TeamID']==team2]
            if len(m1) > 0 and len(m2) > 0:
                massey_diff = m1['CompositeRank'].values[0] - m2['CompositeRank'].values[0]
        
        rows.append({
            'Season'     : season,
            'Team1'      : team1,
            'Team2'      : team2,
            'EloDiff'    : e1['Elo'].values[0]        - e2['Elo'].values[0],
            'WinPctDiff' : s1['WinPct'].values[0]     - s2['WinPct'].values[0],
            'PointDiff'  : s1['PointDiff'].values[0]  - s2['PointDiff'].values[0],
            'SeedDiff'   : seed2['SeedNum'].values[0] - seed1['SeedNum'].values[0],
            'MasseyDiff' : massey_diff,
            'Result'     : result
        })
    
    return pd.DataFrame(rows)

# Build for men
print("\nBuilding men's training data v2...")
m_train_v2 = build_training_data_v2(m_tourney, m_stats, m_elo, m_seeds, massey_by_season)
print(f"✅ Men's training rows: {len(m_train_v2):,}")

# ============================================
# STEP 3 - Check New Feature Correlation
# ============================================
print()
print("=" * 50)
print("FEATURE CORRELATIONS WITH RESULT")
print("=" * 50)
FEATURES_V2 = ['EloDiff', 'WinPctDiff', 'PointDiff', 'SeedDiff', 'MasseyDiff']
corr = m_train_v2[FEATURES_V2 + ['Result']].corr()['Result'].sort_values(ascending=False)
print(corr)

Extracting Massey rankings for all seasons...
✅ Rankings extracted for 24 seasons

Building men's training data v2...
✅ Men's training rows: 2,585

FEATURE CORRELATIONS WITH RESULT
Result        1.000000
SeedDiff      0.492286
MasseyDiff    0.343503
EloDiff       0.340172
PointDiff     0.334501
WinPctDiff    0.323965
Name: Result, dtype: float64


In [19]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import brier_score_loss

FEATURES_V2 = ['EloDiff', 'WinPctDiff', 'PointDiff', 'SeedDiff', 'MasseyDiff']

# ============================================
# TRAIN V2 MODEL
# ============================================
def train_model_v2(train_df, features, gender='Men'):
    X = train_df[features].values
    y = train_df['Result'].values
    
    scaler   = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    
    model = LogisticRegression(C=1.0, max_iter=1000)
    model.fit(X_scaled, y)
    
    y_pred = model.predict_proba(X_scaled)[:, 1]
    brier  = brier_score_loss(y, y_pred)
    
    print(f"{gender}'s Model V2 Results:")
    print(f"  Training Brier Score: {brier:.4f}")
    print(f"\n  Feature Coefficients:")
    for feat, coef in zip(features, model.coef_[0]):
        print(f"    {feat:<15} {coef:+.4f}")
    
    return model, scaler

print("=" * 50)
print("TRAINING V2 MODELS")
print("=" * 50)
m_model_v2, m_scaler_v2 = train_model_v2(m_train_v2, FEATURES_V2, 'Men')

# ============================================
# BUILD WOMEN'S V2 TRAINING DATA
# ============================================
# Note: Massey ordinals are MEN ONLY
# For women we keep V1 features
print()
print("Women's model stays V1 (Massey is men's only)")
print(f"  Women's Brier: 0.1421 (unchanged)")

# ============================================
# CROSS VALIDATE V2
# ============================================
print()
print("=" * 50)
print("CROSS VALIDATION - V1 vs V2 COMPARISON")
print("=" * 50)

def cross_validate_v2(train_df, features, gender='Men'):
    seasons      = sorted(train_df['Season'].unique())
    test_seasons = seasons[5:]
    brier_scores = []
    
    for test_season in test_seasons:
        train_data = train_df[train_df['Season'] <  test_season]
        test_data  = train_df[train_df['Season'] == test_season]
        
        if len(test_data) == 0:
            continue
        
        X_train = train_data[features].values
        y_train = train_data['Result'].values
        X_test  = test_data[features].values
        y_test  = test_data['Result'].values
        
        scaler         = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_test_scaled  = scaler.transform(X_test)
        
        model = LogisticRegression(C=1.0, max_iter=1000)
        model.fit(X_train_scaled, y_train)
        
        y_pred = model.predict_proba(X_test_scaled)[:, 1]
        brier  = brier_score_loss(y_test, y_pred)
        brier_scores.append({'Season': test_season, 'Brier': brier})
    
    return pd.DataFrame(brier_scores)

# Run both
FEATURES_V1 = ['EloDiff', 'WinPctDiff', 'PointDiff', 'SeedDiff']

m_cv_v1 = cross_validate_v2(m_train_v2, FEATURES_V1, 'Men')
m_cv_v2 = cross_validate_v2(m_train_v2, FEATURES_V2, 'Men')

# Compare side by side
print(f"\n{'Season':<10} {'V1 Brier':>10} {'V2 Brier':>10} {'Improvement':>12}")
print("-" * 45)

m_cv_v1 = m_cv_v1.set_index('Season')
m_cv_v2 = m_cv_v2.set_index('Season')

for season in m_cv_v2.index:
    v1 = m_cv_v1.loc[season, 'Brier'] if season in m_cv_v1.index else None
    v2 = m_cv_v2.loc[season, 'Brier']
    
    if v1:
        improvement = v1 - v2
        flag = '✅' if improvement > 0 else '❌'
        print(f"  {int(season):<8} {v1:>10.4f} {v2:>10.4f} {improvement:>+11.4f} {flag}")
    else:
        print(f"  {int(season):<8} {'N/A':>10} {v2:>10.4f} {'N/A':>12}")

print("-" * 45)
print(f"  {'Average':<8} {m_cv_v1['Brier'].mean():>10.4f} {m_cv_v2['Brier'].mean():>10.4f} {m_cv_v1['Brier'].mean()-m_cv_v2['Brier'].mean():>+11.4f}")

TRAINING V2 MODELS
Men's Model V2 Results:
  Training Brier Score: 0.1861

  Feature Coefficients:
    EloDiff         +0.1655
    WinPctDiff      +0.0854
    PointDiff       +0.2998
    SeedDiff        +0.8731
    MasseyDiff      +0.1914

Women's model stays V1 (Massey is men's only)
  Women's Brier: 0.1421 (unchanged)

CROSS VALIDATION - V1 vs V2 COMPARISON

Season       V1 Brier   V2 Brier  Improvement
---------------------------------------------
  1990         0.1900     0.1900     +0.0000 ❌
  1991         0.1839     0.1839     +0.0000 ❌
  1992         0.1715     0.1715     +0.0000 ❌
  1993         0.1672     0.1672     +0.0000 ❌
  1994         0.1721     0.1721     +0.0000 ❌
  1995         0.1776     0.1776     +0.0000 ❌
  1996         0.1754     0.1754     +0.0000 ❌
  1997         0.1775     0.1775     +0.0000 ❌
  1998         0.1993     0.1993     +0.0000 ❌
  1999         0.1937     0.1937     +0.0000 ❌
  2000         0.1804     0.1804     +0.0000 ❌
  2001         0.1903     0.

In [20]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import brier_score_loss
from sklearn.calibration import CalibratedClassifierCV
import xgboost as xgb
import numpy as np
import warnings
warnings.filterwarnings('ignore')

FEATURES_V2 = ['EloDiff', 'WinPctDiff', 'PointDiff', 'SeedDiff', 'MasseyDiff']
FEATURES_V1 = ['EloDiff', 'WinPctDiff', 'PointDiff', 'SeedDiff']

# ============================================
# CROSS VALIDATION FOR ALL MODELS
# ============================================
def cross_validate_models(train_df, features):
    """
    Tests 3 models side by side:
    1. Logistic Regression
    2. XGBoost
    3. Ensemble (average of both)
    """
    seasons      = sorted(train_df['Season'].unique())
    test_seasons = seasons[5:]
    
    results = []
    
    for test_season in test_seasons:
        train_data = train_df[train_df['Season'] <  test_season]
        test_data  = train_df[train_df['Season'] == test_season]
        
        if len(test_data) == 0:
            continue
        
        X_train = train_data[features].values
        y_train = train_data['Result'].values
        X_test  = test_data[features].values
        y_test  = test_data['Result'].values
        
        # --- MODEL 1: Logistic Regression ---
        scaler         = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_test_scaled  = scaler.transform(X_test)
        
        lr = LogisticRegression(C=1.0, max_iter=1000)
        lr.fit(X_train_scaled, y_train)
        pred_lr = lr.predict_proba(X_test_scaled)[:, 1]
        
        # --- MODEL 2: XGBoost ---
        # Key parameters explained:
        # n_estimators  = how many trees to build
        # max_depth      = how deep each tree goes
        # learning_rate  = how much each tree corrects the last
        # subsample      = use 80% of data per tree (prevents overfitting)
        xgb_model = xgb.XGBClassifier(
            n_estimators  = 200,
            max_depth      = 3,
            learning_rate  = 0.05,
            subsample      = 0.8,
            colsample_bytree = 0.8,
            use_label_encoder = False,
            eval_metric    = 'logloss',
            random_state   = 42
        )
        xgb_model.fit(X_train, y_train)
        pred_xgb = xgb_model.predict_proba(X_test)[:, 1]
        
        # --- MODEL 3: Ensemble ---
        # Simple average — you can tune weights later
        # 40% LR + 60% XGB (XGB usually stronger)
        pred_ensemble = 0.4 * pred_lr + 0.6 * pred_xgb
        
        # Clip all predictions to avoid extremes
        pred_lr       = np.clip(pred_lr,       0.025, 0.975)
        pred_xgb      = np.clip(pred_xgb,      0.025, 0.975)
        pred_ensemble = np.clip(pred_ensemble,  0.025, 0.975)
        
        results.append({
            'Season'   : test_season,
            'LR'       : brier_score_loss(y_test, pred_lr),
            'XGB'      : brier_score_loss(y_test, pred_xgb),
            'Ensemble' : brier_score_loss(y_test, pred_ensemble)
        })
    
    return pd.DataFrame(results)

# ============================================
# RUN COMPARISON
# ============================================
print("Running model comparison (this takes ~30 seconds)...")
print()

m_comparison = cross_validate_models(m_train_v2, FEATURES_V2)

print("=" * 60)
print("MEN'S MODEL COMPARISON - Cross Validation Brier Scores")
print("=" * 60)
print(f"\n{'Season':<10} {'Log Reg':>10} {'XGBoost':>10} {'Ensemble':>10} {'Best':>8}")
print("-" * 55)

for _, row in m_comparison.iterrows():
    best = min(row['LR'], row['XGB'], row['Ensemble'])
    flag = ('LR' if best == row['LR'] 
            else 'XGB' if best == row['XGB'] 
            else 'ENS')
    print(f"  {int(row['Season']):<8} {row['LR']:>10.4f} {row['XGB']:>10.4f} {row['Ensemble']:>10.4f} {flag:>8}")

print("-" * 55)
print(f"  {'Average':<8} {m_comparison['LR'].mean():>10.4f} {m_comparison['XGB'].mean():>10.4f} {m_comparison['Ensemble'].mean():>10.4f}")
print()

best_model = min(
    ('LogReg',   m_comparison['LR'].mean()),
    ('XGBoost',  m_comparison['XGB'].mean()),
    ('Ensemble', m_comparison['Ensemble'].mean()),
    key=lambda x: x[1]
)
print(f"🏆 Best model: {best_model[0]} with avg Brier = {best_model[1]:.4f}")

# ============================================
# FEATURE IMPORTANCE FROM XGBOOST
# ============================================
print()
print("=" * 50)
print("XGBOOST FEATURE IMPORTANCE")
print("=" * 50)
print("(Which features does XGBoost rely on most?)")
print()

# Train final XGB on all data to get feature importance
scaler_final   = StandardScaler()
X_all          = m_train_v2[FEATURES_V2].values
y_all          = m_train_v2['Result'].values

xgb_final = xgb.XGBClassifier(
    n_estimators     = 200,
    max_depth        = 3,
    learning_rate    = 0.05,
    subsample        = 0.8,
    colsample_bytree = 0.8,
    use_label_encoder = False,
    eval_metric      = 'logloss',
    random_state     = 42
)
xgb_final.fit(X_all, y_all)

importance = xgb_final.feature_importances_
for feat, imp in sorted(zip(FEATURES_V2, importance), key=lambda x: -x[1]):
    bar = '█' * int(imp * 50)
    print(f"  {feat:<15} {imp:.4f}  {bar}")


Running model comparison (this takes ~30 seconds)...

MEN'S MODEL COMPARISON - Cross Validation Brier Scores

Season        Log Reg    XGBoost   Ensemble     Best
-------------------------------------------------------
  1990         0.1900     0.1877     0.1854      ENS
  1991         0.1839     0.1951     0.1847       LR
  1992         0.1715     0.2063     0.1879       LR
  1993         0.1672     0.1542     0.1559      XGB
  1994         0.1721     0.1934     0.1804       LR
  1995         0.1776     0.1936     0.1843       LR
  1996         0.1754     0.1748     0.1718      ENS
  1997         0.1775     0.1963     0.1855       LR
  1998         0.1993     0.1959     0.1947      ENS
  1999         0.1937     0.1843     0.1847      XGB
  2000         0.1804     0.1745     0.1743      ENS
  2001         0.1903     0.1780     0.1802      XGB
  2002         0.1955     0.1986     0.1953      ENS
  2003         0.1857     0.2035     0.1938       LR
  2004         0.1693     0.1811     0.

In [21]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import brier_score_loss
from sklearn.calibration import CalibratedClassifierCV
import xgboost as xgb
import numpy as np
import warnings
warnings.filterwarnings('ignore')

FEATURES_V2 = ['EloDiff', 'WinPctDiff', 'PointDiff', 'SeedDiff', 'MasseyDiff']
FEATURES_V1 = ['EloDiff', 'WinPctDiff', 'PointDiff', 'SeedDiff']

# ============================================
# CROSS VALIDATION FOR ALL MODELS
# ============================================
def cross_validate_models(train_df, features):
    """
    Tests 3 models side by side:
    1. Logistic Regression
    2. XGBoost
    3. Ensemble (average of both)
    """
    seasons      = sorted(train_df['Season'].unique())
    test_seasons = seasons[5:]
    
    results = []
    
    for test_season in test_seasons:
        train_data = train_df[train_df['Season'] <  test_season]
        test_data  = train_df[train_df['Season'] == test_season]
        
        if len(test_data) == 0:
            continue
        
        X_train = train_data[features].values
        y_train = train_data['Result'].values
        X_test  = test_data[features].values
        y_test  = test_data['Result'].values
        
        # --- MODEL 1: Logistic Regression ---
        scaler         = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_test_scaled  = scaler.transform(X_test)
        
        lr = LogisticRegression(C=1.0, max_iter=1000)
        lr.fit(X_train_scaled, y_train)
        pred_lr = lr.predict_proba(X_test_scaled)[:, 1]
        
        # --- MODEL 2: XGBoost ---
        # Key parameters explained:
        # n_estimators  = how many trees to build
        # max_depth      = how deep each tree goes
        # learning_rate  = how much each tree corrects the last
        # subsample      = use 80% of data per tree (prevents overfitting)
        xgb_model = xgb.XGBClassifier(
            n_estimators  = 200,
            max_depth      = 3,
            learning_rate  = 0.05,
            subsample      = 0.8,
            colsample_bytree = 0.8,
            use_label_encoder = False,
            eval_metric    = 'logloss',
            random_state   = 42
        )
        xgb_model.fit(X_train, y_train)
        pred_xgb = xgb_model.predict_proba(X_test)[:, 1]
        
        # --- MODEL 3: Ensemble ---
        # Simple average — you can tune weights later
        # 40% LR + 60% XGB (XGB usually stronger)
        pred_ensemble = 0.4 * pred_lr + 0.6 * pred_xgb
        
        # Clip all predictions to avoid extremes
        pred_lr       = np.clip(pred_lr,       0.025, 0.975)
        pred_xgb      = np.clip(pred_xgb,      0.025, 0.975)
        pred_ensemble = np.clip(pred_ensemble,  0.025, 0.975)
        
        results.append({
            'Season'   : test_season,
            'LR'       : brier_score_loss(y_test, pred_lr),
            'XGB'      : brier_score_loss(y_test, pred_xgb),
            'Ensemble' : brier_score_loss(y_test, pred_ensemble)
        })
    
    return pd.DataFrame(results)

# ============================================
# RUN COMPARISON
# ============================================
print("Running model comparison (this takes ~30 seconds)...")
print()

m_comparison = cross_validate_models(m_train_v2, FEATURES_V2)

print("=" * 60)
print("MEN'S MODEL COMPARISON - Cross Validation Brier Scores")
print("=" * 60)
print(f"\n{'Season':<10} {'Log Reg':>10} {'XGBoost':>10} {'Ensemble':>10} {'Best':>8}")
print("-" * 55)

for _, row in m_comparison.iterrows():
    best = min(row['LR'], row['XGB'], row['Ensemble'])
    flag = ('LR' if best == row['LR'] 
            else 'XGB' if best == row['XGB'] 
            else 'ENS')
    print(f"  {int(row['Season']):<8} {row['LR']:>10.4f} {row['XGB']:>10.4f} {row['Ensemble']:>10.4f} {flag:>8}")

print("-" * 55)
print(f"  {'Average':<8} {m_comparison['LR'].mean():>10.4f} {m_comparison['XGB'].mean():>10.4f} {m_comparison['Ensemble'].mean():>10.4f}")
print()

best_model = min(
    ('LogReg',   m_comparison['LR'].mean()),
    ('XGBoost',  m_comparison['XGB'].mean()),
    ('Ensemble', m_comparison['Ensemble'].mean()),
    key=lambda x: x[1]
)
print(f"🏆 Best model: {best_model[0]} with avg Brier = {best_model[1]:.4f}")

# ============================================
# FEATURE IMPORTANCE FROM XGBOOST
# ============================================
print()
print("=" * 50)
print("XGBOOST FEATURE IMPORTANCE")
print("=" * 50)
print("(Which features does XGBoost rely on most?)")
print()

# Train final XGB on all data to get feature importance
scaler_final   = StandardScaler()
X_all          = m_train_v2[FEATURES_V2].values
y_all          = m_train_v2['Result'].values

xgb_final = xgb.XGBClassifier(
    n_estimators     = 200,
    max_depth        = 3,
    learning_rate    = 0.05,
    subsample        = 0.8,
    colsample_bytree = 0.8,
    use_label_encoder = False,
    eval_metric      = 'logloss',
    random_state     = 42
)
xgb_final.fit(X_all, y_all)

importance = xgb_final.feature_importances_
for feat, imp in sorted(zip(FEATURES_V2, importance), key=lambda x: -x[1]):
    bar = '█' * int(imp * 50)
    print(f"  {feat:<15} {imp:.4f}  {bar}")


Running model comparison (this takes ~30 seconds)...

MEN'S MODEL COMPARISON - Cross Validation Brier Scores

Season        Log Reg    XGBoost   Ensemble     Best
-------------------------------------------------------
  1990         0.1900     0.1877     0.1854      ENS
  1991         0.1839     0.1951     0.1847       LR
  1992         0.1715     0.2063     0.1879       LR
  1993         0.1672     0.1542     0.1559      XGB
  1994         0.1721     0.1934     0.1804       LR
  1995         0.1776     0.1936     0.1843       LR
  1996         0.1754     0.1748     0.1718      ENS
  1997         0.1775     0.1963     0.1855       LR
  1998         0.1993     0.1959     0.1947      ENS
  1999         0.1937     0.1843     0.1847      XGB
  2000         0.1804     0.1745     0.1743      ENS
  2001         0.1903     0.1780     0.1802      XGB
  2002         0.1955     0.1986     0.1953      ENS
  2003         0.1857     0.2035     0.1938       LR
  2004         0.1693     0.1811     0.

In [22]:
print("\nBuilding training data...")

def build_training_data(tourney_df, stats_df, elo_df, seeds_df, 
                         massey_by_season=None):
    rows = []
    for _, game in tourney_df.iterrows():
        season = game['Season']
        wteam  = game['WTeamID']
        lteam  = game['LTeamID']

        if wteam < lteam:
            team1, team2, result = wteam, lteam, 1
        else:
            team1, team2, result = lteam, wteam, 0

        s1    = stats_df[(stats_df['Season']==season) & (stats_df['TeamID']==team1)]
        s2    = stats_df[(stats_df['Season']==season) & (stats_df['TeamID']==team2)]
        e1    = elo_df[elo_df['TeamID']==team1]
        e2    = elo_df[elo_df['TeamID']==team2]
        seed1 = seeds_df[(seeds_df['Season']==season) & (seeds_df['TeamID']==team1)]
        seed2 = seeds_df[(seeds_df['Season']==season) & (seeds_df['TeamID']==team2)]

        if any(len(x)==0 for x in [s1, s2, e1, e2, seed1, seed2]):
            continue

        massey_diff = 0
        if massey_by_season and season in massey_by_season:
            m_ranks = massey_by_season[season]
            m1 = m_ranks[m_ranks['TeamID']==team1]
            m2 = m_ranks[m_ranks['TeamID']==team2]
            if len(m1) > 0 and len(m2) > 0:
                massey_diff = m1['CompositeRank'].values[0] - m2['CompositeRank'].values[0]

        rows.append({
            'Season'     : season,
            'Team1'      : team1,
            'Team2'      : team2,
            'EloDiff'    : e1['Elo'].values[0]        - e2['Elo'].values[0],
            'WinPctDiff' : s1['WinPct'].values[0]     - s2['WinPct'].values[0],
            'PointDiff'  : s1['PointDiff'].values[0]  - s2['PointDiff'].values[0],
            'SeedDiff'   : seed2['SeedNum'].values[0] - seed1['SeedNum'].values[0],
            'MasseyDiff' : massey_diff,
            'Result'     : result
        })

    return pd.DataFrame(rows)

m_train = build_training_data(m_tourney, m_stats, m_elo, m_seeds, massey_by_season)
w_train = build_training_data(w_tourney, w_stats, w_elo, w_seeds)

print(f"✅ Men's training rows:   {len(m_train):,}")
print(f"✅ Women's training rows: {len(w_train):,}")





Building training data...
✅ Men's training rows:   2,585
✅ Women's training rows: 1,717


In [23]:
# ============================================
# STEP 7 - TRAIN FINAL MODELS
# ============================================
print("\nTraining final models...")

M_FEATURES = ['EloDiff', 'WinPctDiff', 'PointDiff', 'SeedDiff', 'MasseyDiff']
W_FEATURES = ['EloDiff', 'WinPctDiff', 'PointDiff', 'SeedDiff']

def train_final_ensemble(train_df, features):
    X = train_df[features].values
    y = train_df['Result'].values

    # Logistic Regression
    scaler   = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    lr       = LogisticRegression(C=1.0, max_iter=1000)
    lr.fit(X_scaled, y)

    # XGBoost
    xgb_model = xgb.XGBClassifier(
        n_estimators     = 200,
        max_depth        = 3,
        learning_rate    = 0.05,
        subsample        = 0.8,
        colsample_bytree = 0.8,
        eval_metric      = 'logloss',
        random_state     = 42
    )
    xgb_model.fit(X, y)

    # Brier scores
    pred_lr  = np.clip(lr.predict_proba(X_scaled)[:,1],   0.025, 0.975)
    pred_xgb = np.clip(xgb_model.predict_proba(X)[:,1],   0.025, 0.975)
    pred_ens = np.clip(0.4*pred_lr + 0.6*pred_xgb,        0.025, 0.975)

    print(f"  LR Brier:       {brier_score_loss(y, pred_lr):.4f}")
    print(f"  XGB Brier:      {brier_score_loss(y, pred_xgb):.4f}")
    print(f"  Ensemble Brier: {brier_score_loss(y, pred_ens):.4f}")

    return lr, scaler, xgb_model

print("\nMen's models:")
m_lr, m_scaler, m_xgb = train_final_ensemble(m_train, M_FEATURES)

print("\nWomen's models:")
w_lr, w_scaler, w_xgb = train_final_ensemble(w_train, W_FEATURES)




Training final models...

Men's models:
  LR Brier:       0.1861
  XGB Brier:      0.1617
  Ensemble Brier: 0.1700

Women's models:
  LR Brier:       0.1422
  XGB Brier:      0.1203
  Ensemble Brier: 0.1278


In [ ]:
# ============================================
# STEP 8 - GENERATE SUBMISSION
# ============================================
print("\nGenerating predictions for all 2026 matchups...")

predictions = []
missing     = 0

for _, row in submission.iterrows():
    parts  = row['ID'].split('_')
    season = int(parts[0])
    team1  = int(parts[1])
    team2  = int(parts[2])

    is_mens = team1 < 2000

    if is_mens:
        stats_df = m_stats
        elo_df   = m_elo
        seeds_df = m_seeds
        features = M_FEATURES
        lr       = m_lr
        scaler   = m_scaler
        xgb_model= m_xgb
    else:
        stats_df = w_stats
        elo_df   = w_elo
        seeds_df = w_seeds
        features = W_FEATURES
        lr       = w_lr
        scaler   = w_scaler
        xgb_model= w_xgb

    s1    = stats_df[(stats_df['Season']==season) & (stats_df['TeamID']==team1)]
    s2    = stats_df[(stats_df['Season']==season) & (stats_df['TeamID']==team2)]
    e1    = elo_df[elo_df['TeamID']==team1]
    e2    = elo_df[elo_df['TeamID']==team2]
    seed1 = seeds_df[(seeds_df['Season']==season) & (seeds_df['TeamID']==team1)]
    seed2 = seeds_df[(seeds_df['Season']==season) & (seeds_df['TeamID']==team2)]

    if any(len(x)==0 for x in [s1, s2, e1, e2]):
        predictions.append(0.5)
        missing += 1
        continue

    s1_num = seed1['SeedNum'].values[0] if len(seed1) > 0 else 8.5
    s2_num = seed2['SeedNum'].values[0] if len(seed2) > 0 else 8.5

    massey_diff = 0
    if is_mens and season in massey_by_season:
        m_ranks = massey_by_season[season]
        m1 = m_ranks[m_ranks['TeamID']==team1]
        m2 = m_ranks[m_ranks['TeamID']==team2]
        if len(m1) > 0 and len(m2) > 0:
            massey_diff = m1['CompositeRank'].values[0] - m2['CompositeRank'].values[0]

    feat_dict = {
        'EloDiff'    : e1['Elo'].values[0]        - e2['Elo'].values[0],
        'WinPctDiff' : s1['WinPct'].values[0]     - s2['WinPct'].values[0],
        'PointDiff'  : s1['PointDiff'].values[0]  - s2['PointDiff'].values[0],
        'SeedDiff'   : s2_num                     - s1_num,
        'MasseyDiff' : massey_diff
    }

    feat_row = pd.DataFrame([[feat_dict[f] for f in features]], columns=features)

    # Ensemble prediction
    pred_lr  = lr.predict_proba(scaler.transform(feat_row))[:,1][0]
    pred_xgb = xgb_model.predict_proba(feat_row.values)[:,1][0]
    prob     = np.clip(0.4*pred_lr + 0.6*pred_xgb, 0.025, 0.975)

    predictions.append(prob)



Generating predictions for all 2026 matchups...


In [ ]:
# ============================================
# STEP 9 - SAVE SUBMISSION
# ============================================
submission['Pred'] = predictions
final = submission[['ID','Pred']]
final.to_csv('/kaggle/working/submission.csv', index=False)

print()
print("=" * 50)
print("SUBMISSION SUMMARY")
print("=" * 50)
print(f"Total rows:        {len(final):,}")
print(f"Missing/defaulted: {missing:,}")
print(f"Min prediction:    {final['Pred'].min():.4f}")
print(f"Max prediction:    {final['Pred'].max():.4f}")
print(f"Mean prediction:   {final['Pred'].mean():.4f}")
print()
print("Sample predictions:")
print(final.head(10).to_string(index=False))
print()
print("✅ submission.csv saved to /kaggle/working/")
print("   Download it and submit to Kaggle!")